# 05 — Writing a Custom Experiment

Every experiment in `reconstruct/experiments/` follows the same two-class pattern:

```
MyProgram(BaseProgram)      ← defines the QICK pulse sequence
MyExperiment(BaseExperiment) ← manages sweep, acquisition, fitting, saving
```

This notebook builds a **Spin Echo** experiment from scratch to illustrate the pattern.
The same code structure is used by every built-in experiment.

In [ ]:
import sys; sys.path.insert(0, '../')
import numpy as np
import matplotlib.pyplot as plt

from reconstruct import SimulatedBackend, ExperimentConfig
from reconstruct.core.base_program    import BaseProgram
from reconstruct.core.base_experiment import BaseExperiment
from reconstruct.core.experiment_data import ExperimentData, QualityFlag

backend = SimulatedBackend(noise_level=0.02)
backend.activate()

config_list = [{
    "name": "Q1",
    "ch":  {"ro_ch": 0, "res_ch": 0, "qb_ch": 1},
    "res": {"res_freq_ge": 6700.0, "res_gain": 0.5, "res_length": 2.0},
    "qb":  {"qb_freq_ge": 5000.0, "pi_gain_ge": 0.5, "sigma": 0.025},
    "reps": 200, "relax_delay": 300.0, "steps": 51,
    "wait_time_start": 0.0, "wait_time_stop": 100.0,
    "T2_true": 25.0,
}]
cfg_all = ExperimentConfig(config_list)
cfg = cfg_all.get_qubit('Q1')

## Step 1 — Define the QICK program

`_initialize(cfg)` declares channels and the sweep loop.  
`_body(cfg)` is the pulse sequence executed for each loop iteration.

Key `BaseProgram` helpers you can call inside `_initialize` / `_body`:

| Method | What it does |
|---|---|
| `setup_resonator(cfg)` | Declare readout channel and pulse |
| `setup_qubit_gen(cfg, prefix)` | Declare qubit drive generator |
| `setup_standard_gates(cfg, prefix)` | Register x180, y180, x90, x90m, y90, y90m |
| `add_loop(name, n)` | Add a sweep loop of `n` steps |
| `pulse(ch, name, t)` | Fire a named pulse |
| `delay_auto(dt)` | Auto-delay between pulses |
| `measure(cfg)` | Trigger readout ADC |

In [ ]:
class SpinEchoProgram(BaseProgram):
    """π/2 — wait τ/2 — π — wait τ/2 — π/2 — readout.  Sweeps τ."""

    def _initialize(self, cfg):
        self.setup_resonator(cfg)
        self.setup_qubit_gen(cfg, prefix='ge')
        self.setup_standard_gates(cfg, prefix='ge')
        # One loop variable: the free-evolution time τ
        self.add_loop('timeloop', cfg['steps'])

    def _body(self, cfg):
        self.send_readoutconfig(ch=cfg['ro_ch'], name='myro', t=0)
        # π/2 pulse
        self.pulse(ch=cfg['qb_ch'], name='x90_ge',  t=0)
        self.delay_auto(0.01)
        # Refocusing π pulse at half the wait time
        self.pulse(ch=cfg['qb_ch'], name='x180_ge', t=0)
        self.delay_auto(0.01)
        # Second π/2 pulse
        self.pulse(ch=cfg['qb_ch'], name='x90_ge',  t=0)
        self.delay_auto(0.02)
        self.measure(cfg)

## Step 2 — Define the experiment class

The mandatory class attributes tell the framework how to display and save results.  
Override `_post_fit` to add your own fitting logic after the acquisition.

In [ ]:
from scipy.optimize import curve_fit

class SpinEcho(BaseExperiment):
    """Spin Echo — extracts T2 echo coherence time."""

    EXPT_NAME    = 'custom_spin_echo'
    TAG          = 'Coherence'
    X_LABEL      = 'Wait time (µs)'
    TITLE_PREFIX = 'Spin Echo'
    SWEEP_KEYS_TO_REMOVE = ['wait_time_start', 'wait_time_stop']
    X_SAVE_NAME  = 'Time'
    X_SAVE_UNIT  = 'us'
    X_SAVE_SCALE = 1.0

    def _create_program(self):
        return SpinEchoProgram(
            self.soccfg,
            reps=self.cfg['reps'],
            final_delay=self.cfg['relax_delay'],
            cfg=self.cfg,
        )

    def _extract_sweep_axis(self, prog):
        return prog.get_time_param('timeloop', 't', as_array=True)

    def _post_fit(self, x_vals):
        """Fit exponential decay A*exp(-t/T2e) + C."""
        y = self.iqdata
        try:
            def model(t, A, T2, C):
                return A * np.exp(-t / T2) + C

            p0 = [y[0] - y[-1], x_vals[-1] / 3, y[-1]]
            popt, pcov = curve_fit(model, x_vals, y, p0=p0, maxfev=4000)
            perr = np.sqrt(np.diag(pcov))

            self.fit_params = popt
            self.fit_errors = perr
            self.param = {
                'T2e_us': (round(popt[1], 2), round(perr[1], 2)),
            }
            return self.param
        except Exception as e:
            print(f'SpinEcho fit failed: {e}')
            self.param = None
            return None

## Step 3 — Run it

In [ ]:
expt = SpinEcho(cfg, backend=backend)
result = expt.run(py_avg=8)

print('Quality     :', result.quality)
print('fit_result  :', result.fit_result)
print('T2e         :', result.scalar_result)

In [ ]:
# Plot raw data + exponential fit overlay
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(result.x_axis, result.y_axis, 'o', ms=3, label='data')

if expt.fit_params is not None:
    t_fit = np.linspace(result.x_axis[0], result.x_axis[-1], 300)
    A, T2, C = expt.fit_params
    ax.plot(t_fit, A * np.exp(-t_fit / T2) + C, 'r-', lw=1.5,
            label=f'T2e = {T2:.1f} µs')

ax.set_xlabel('Wait time (µs)')
ax.set_ylabel('Signal (a.u.)')
ax.set_title('Custom Spin Echo')
ax.legend()
plt.tight_layout()
plt.show()

## Where to put the new experiment

To make it part of the package:

1. Save the two classes in `reconstruct/experiments/coherence/spin_echo.py`
2. Export from `reconstruct/experiments/coherence/__init__.py`:
   ```python
   from .spin_echo import SpinEcho
   ```
3. Import paths inside `reconstruct/experiments/coherence/` are **3 dots** deep:
   ```python
   from ...core.base_program    import BaseProgram
   from ...core.base_experiment import BaseExperiment
   from ...tools.fitting        import fitdecaysin
   ```

**Never import from `qick_workspace`** — all utilities live in `reconstruct/tools/`.

**Next:** [06_real_hardware.ipynb](06_real_hardware.ipynb) — connecting to QICK hardware, saving data, and running the REST service.